# Sección II.A de Polonyi (2019), en modo exploración

Las figuras de `seccion_A/` son el **producto final**: guardan un PNG y no se ven
en el notebook (usan el backend `Agg`, sin pantalla). Este notebook hace lo
contrario: arma cada gráfica **aquí**, importando de `nlaid/`, para que puedas
cambiar un número y volver a mirar.

| | qué es | cómo se usa |
|---|---|---|
| `nlaid/` | la **herramienta** | se importa: `from nlaid.core import ...` |
| `scripts/`, `seccion_A/` | el **producto** | se corren: `%run ...` o `python ...` |

Recorrido: ec. (7) → ec. (10) → ec. (14) → ec. (15) → ec. (16).

In [ ]:
# === 0. SETUP ============================================================
# Funciona en local (desde el repo) y en Google Colab (clona si hace falta).
import importlib.util, pathlib, subprocess, sys

REPO = "https://github.com/JoMZ-ops/Non-Local-AID-thesis"
RAMA = "Electrodynamics---Abraham-&-Lorentz-Force"

if importlib.util.find_spec("nlaid") is None:
    raiz = pathlib.Path.cwd()
    if not (raiz / "nlaid").is_dir():
        raiz = raiz.parent
    if (raiz / "nlaid").is_dir():
        sys.path.insert(0, str(raiz))            # local: el repo es el padre
    else:
        subprocess.run(["pip", "install", "-q", "numpy", "scipy", "matplotlib"])
        subprocess.run(["git", "clone", "-q", "--branch", RAMA, REPO])
        sys.path.insert(0, "Non-Local-AID-thesis")

import nlaid
print("nlaid importado desde:", nlaid.RAIZ)

In [ ]:
# === 1. Imports ==========================================================
%matplotlib inline
%load_ext autoreload
%autoreload 2
# autoreload: si editas un archivo de nlaid/, la proxima celda ya usa la
# version nueva -- sin reiniciar el kernel ni reinstalar nada.

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

from nlaid.core import Params, make_regulator
from nlaid.block1_linear import (susceptibility, find_zeros_uhp,
                                 convergence_lower_bound)
from nlaid.block2_delay import integrate_delay
from nlaid.worldline import smooth_bump

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": .3, "font.size": 10})

chi = lambda w, x: susceptibility(
    w, make_regulator("smeared", 1 / x), Params(ell=1 / x))   # x = r0/ell

## Ec. (7) — el núcleo de la ecuación linealizada

$$\ddot x = 4 r_{0B} \int_{-\infty}^{0} du\; \delta_B'(u^2)\,(x - x' + u\,\dot x')$$

No es una curva: es una integral. Lo que se puede mirar es su **integrando**, y
lo que gobierna la memoria es el núcleo $4 r_0 \delta_B'(u^2)$.

Dos cosas que leer en la gráfica:

- cambia de signo en $u = -2\ell$ (donde $1 - \sqrt{z}/2\ell$ se anula);
- al apretar el cutoff se **angosta como $\ell$ y crece como $\ell^{-4}$**.

In [ ]:
u = np.linspace(-2.0, 0, 800)

for x in (2, 3, 4):                       # x = r0/ell
    ell = 1 / x
    reg = make_regulator("smeared", ell)
    plt.plot(u, 4 * reg.d_delta(u ** 2), lw=2, label=f"$r_0/\\ell$ = {x}")
    plt.plot([-2 * ell], [0], "o", ms=5)  # el cambio de signo, en u = -2l

plt.yscale("symlog", linthresh=1)         # los dos lobulos difieren x40
plt.axhline(0, color="k", lw=.8)
plt.xlabel("$u = s' - s$"); plt.ylabel("$4 r_0 \\delta_B'(u^2)$")
plt.legend(); plt.show()

**Pruébalo:** cambia `(2, 3, 4)` por `(1, 2, 8)` y vuelve a correr la celda. El
núcleo se concentra cada vez más cerca de $u = 0$: eso es la memoria
encogiéndose con el cutoff.

## Ec. (10) — el contratérmino de masa

$$\delta m = \frac{e^2}{2c^2}\int_0^\infty \frac{dz}{\sqrt z}\,\delta_B(z)$$

Con la ec. (5) y el cambio $z = \ell^2 t^2$ sale $\delta m/m = r_0/(6\ell)$.
La regla del repo es **validar cada número por una vía independiente**: aquí van
tres (cuadratura, forma cerrada y el propio módulo) que deben coincidir.

In [ ]:
ell = 0.7
reg = make_regulator("smeared", ell)

print("cuadratura         ", quad(lambda z: reg.delta(z) / np.sqrt(z), 0, np.inf, limit=200)[0])
print("cerrada 1/(3 ell)  ", 1 / (3 * ell))
print("modulo             ", reg.moment_inv_sqrt())
print("delta_m/m          ", reg.mass_shift_over_m(), " = r0/(6 ell) =", 1 / (6 * ell))

# El integrando: su area ES el contratermino.
z = np.linspace(1e-9, 60, 800)
g = reg.delta(z * ell ** 2) / np.sqrt(z * ell ** 2) * ell    # adimensional
plt.fill_between(z, 0, g, alpha=.3); plt.plot(z, g, lw=2)
plt.xlabel("$z/\\ell^2$"); plt.ylabel("$\\ell\\, z^{-1/2}\\delta_B(z)$")
plt.title("área = 1/3   $\\Rightarrow$   $\\delta m/m = r_0/6\\ell$")
plt.show()

## Ec. (14) — la susceptibilidad, y sus ceros

$$F^r_\omega = \frac{1}{(\omega + i\epsilon)^2\, \chi^r_\omega}$$

Los **polos de la respuesta son los ceros de $\chi$**. Un cero con
$\operatorname{Im}\omega > 0$ es un modo $e^{-i\omega s}$ que crece.

El mapa de $|\chi|$ tarda unos segundos: baja `n` si quieres iterar más rápido.

In [ ]:
def mapa(x, n=140, re_max=8, im_hi=5):
    # |chi| sobre el plano complejo, evaluado por columnas para no pedir
    # una malla plana de cientos de MB.
    reg, pr = make_regulator("smeared", 1 / x), Params(ell=1 / x)
    im_lo = 0.85 * convergence_lower_bound(reg)      # la integral converge ahi
    R, I = np.meshgrid(np.linspace(-re_max, re_max, n),
                       np.linspace(max(im_lo, -2.5), im_hi, n), indexing="ij")
    A = np.stack([np.abs(susceptibility(col, reg, pr)) for col in R + 1j * I])
    return R, I, np.log10(A + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, x in zip(axes, (3, 5)):
    R, I, A = mapa(x)
    ax.contourf(R, I, A, levels=20, cmap="Blues_r")
    ax.axhline(0, color="k", lw=1.2)
    for w in find_zeros_uhp(make_regulator("smeared", 1 / x), Params(ell=1 / x),
                            re_max=8, im_hi=5, grid=120):
        ax.plot(w.real, w.imag, "x", ms=10, mew=2.5, color="#c1442e")
    ax.set_title(f"$r_0/\\ell$ = {x}"); ax.set_xlabel("Re $\\omega$")
axes[0].set_ylabel("Im $\\omega$")
plt.show()

A `r0/ell = 3` no hay ninguna X: el par dominante está debajo del eje real, la
teoría es estable. A `r0/ell = 5` ya cruzó. El borde está en **`r0/ell = 4.00`**
(`critical_cutoff("smeared")`, que lo calcula por principio del argumento sin
localizar un solo cero).

## Ec. (15) — el límite local

$$\chi^r_\omega = 1 + r_0\,\omega\left[\tfrac{2}{3} i + O(\omega\ell)\right]$$

Es la (14) con el cutoff removido a frecuencia fija. Su único cero está en
$\omega = 3i/2$ **para cualquier $\ell$**: el runaway de Abraham-Lorentz.

Un hecho que hace la comparación limpia: el corchete $(\chi-1)/(r_0\omega)$ es
función de $\omega\ell$ **sola**, así que las curvas de distintos cutoffs
colapsan. Eso vuelve literal el $O(\omega\ell)$.

In [ ]:
for x, lw in zip((2, 3, 4), (6, 3, 1.4)):
    w = np.linspace(0.01, 3, 300) * x                  # omega*ell de 0.01 a 3
    g = (chi(w, x) - 1) / w                            # r0 = 1
    plt.plot(w / x, g.imag, lw=lw, label=f"$r_0/\\ell$ = {x}")

plt.axhline(2 / 3, color="#c1442e", lw=2)
plt.annotate("ec. (15): $2i/3$", (1.6, 0.70), color="#c1442e")
plt.xlabel("$\\omega\\ell$"); plt.ylabel("Im $(\\chi - 1)/r_0\\omega$")
plt.legend(); plt.show()

print("a omega*ell = 0.5, el corchete vale:")
for x in (2, 3, 4):
    print(f"   r0/ell={x}:  {(chi(0.5 * x, x) - 1) / (0.5 * x):.9f}")

Los tres números son idénticos a nueve decimales: el colapso es exacto, no
aproximado. Y la lectura práctica: la ec. (15) vale para $\omega \ll 1/\ell$,
pero el modo que decide la estabilidad vive en $\omega\ell = 1$ — justo donde
la aproximación local ya no vale.

## Ec. (16) — el retardo finito

El punto retardado $x'$ está donde $\ell^2 = (x-x')^2$: una distancia
**invariante**, no un retardo de coordenada. Esta celda tarda ~15 s.

In [ ]:
ell = 1.0
wl = integrate_delay(Params(ell=ell), s_end=8.0, ds=5e-3,
                     drive=lambda s: smooth_bump(s, amplitude=0.9, width=1.5))

s = np.linspace(-1.5, 7, 150)
ret = np.array([wl.retarded_point(q, wl.sample(q)[0], ell)[0] for q in s])
dif = np.array([wl.sample(q)[0] - wl.sample(r)[0] for q, r in zip(s, ret)])

print("max |(x-x')^2 - ell^2| =", np.abs(dif[:, 0]**2 - dif[:, 1]**2 - ell**2).max())

plt.plot(s, s - ret, lw=2, label="retardo propio $\\Delta s$")
plt.plot(s, dif[:, 0], lw=2, label="retardo de coordenada $\\Delta t$")
plt.plot(s, dif[:, 1], lw=2, ls="--", label="separación espacial $\\Delta x$")
plt.axhline(ell, color="k", lw=.8, ls=":")
plt.xlabel("$s$"); plt.ylabel("en unidades de $\\ell$ = 1")
plt.legend(); plt.show()

El invariante vale $\ell$ en todo $s$ (residuo $\sim 10^{-14}$), pero su reparto
entre $\Delta t$ y $\Delta x$ cambia con el movimiento. Por eso la ec. (16) es
una **DDE de retardo dependiente del estado**, no una EDO.

**Cuidado al comparar bloques:** `susceptibility` (ec. 14) no lee `m_over_mB`,
pero `integrate_delay` (ec. 16) sí multiplica por él. Compararlos fijándose solo
en `r0/ell` da veredictos opuestos con la misma ecuación —
ver `docs/comparabilidad_bloques.md`.

## Reproducir las figuras completas

Los scripts de `seccion_A/` guardan un PNG y **no** lo muestran. Para verlo:

In [ ]:
ruta = f"{nlaid.RAIZ}/seccion_A/fig_ec7_ec10.py"
%run $ruta
from IPython.display import Image
Image(f"{nlaid.RAIZ}/figures/seccionA_ec7_ec10.png")

In [ ]:
# El %run de arriba dejo el backend en Agg y tus graficas inline dejarian de
# aparecer. Esta linea lo devuelve a la normalidad.
%matplotlib inline

## Siguiente paso

- Cambia parámetros en las celdas y vuelve a correr: es para lo que está.
- Para **modificar el código de verdad**, edita `nlaid/*.py` en un clon local
  (con `pip install -e .`); con `autoreload` el cambio se ve sin reiniciar.
- En Colab los archivos son efímeros: sirven para correr, no para editar.

Los cuatro scripts completos: `fig_ec7_ec10.py`, `fig_ec14_polos.py`,
`fig_ec15_vs_ec14.py`, `fig_ec16_retardo.py`.